<a href="https://colab.research.google.com/github/Adhamm03/vllm-qwen-api/blob/main/vllm_qwen_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Free open-source LLM API with vLLM + Qwen2.5-3B-Instruct

Serve an **OpenAI-compatible API** from a free Colab T4 GPU and reach it from anywhere through a Cloudflare quick tunnel.

```
Your app → Cloudflare tunnel (public URL) → vLLM server → Qwen2.5-3B-Instruct on a T4
```

| Tool | Role |
|---|---|
| Qwen2.5-3B-Instruct | The model (weights from Hugging Face) |
| vLLM | Serving engine; exposes `/v1/chat/completions` |
| Google Colab (T4 GPU) | Free GPU |
| cloudflared | Free tunnel that gives the server a public URL |
| OpenAI Python SDK | Client that talks to the server |

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

## 2. Install vLLM
Takes a few minutes. If Colab asks to restart the runtime, do it, then continue from step 3.

In [ ]:
!pip install -q vllm

## 3. Start the vLLM server
- `--dtype half`: the T4 has no bf16 support, so weights run in fp16.
- `--max-model-len 4096`: caps context length so memory isn't exhausted.
- No `--api-key`: the client uses `"EMPTY"`. Add `--api-key <secret>` if you share the URL.

⚠️ **Run this cell once.** Starting a second server while the first is still using the GPU crashes startup (see Troubleshooting). The first start downloads ~6 GB and can take a few minutes.

In [ ]:
!nohup vllm serve Qwen/Qwen2.5-3B-Instruct \
  --dtype half \
  --max-model-len 4096 \
  --gpu-memory-utilization 0.85 \
  --port 8000 > vllm.log 2>&1 &

!for i in $(seq 1 60); do curl -s localhost:8000/v1/models > /dev/null && echo READY && break; sleep 5; done

## 4. Expose it with a Cloudflare quick tunnel
Run this only after the previous cell prints `READY`.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!nohup ./cloudflared tunnel --url http://localhost:8000 > tunnel.log 2>&1 &
!sleep 8 && grep -o "https://[a-z0-9-]*\.trycloudflare\.com" tunnel.log

## 5. Call the API
The URL is read from the tunnel log automatically.

In [ ]:
import re
from openai import OpenAI

url = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read()).group(0)
client = OpenAI(base_url=f"{url}/v1", api_key="EMPTY")

r = client.chat.completions.create(
    model="Qwen/Qwen2.5-3B-Instruct",
    messages=[{"role": "user", "content": "Hello!"}],
)
print(r.choices[0].message.content)

### Streaming

In [ ]:
stream = client.chat.completions.create(
    model="Qwen/Qwen2.5-3B-Instruct",
    messages=[{"role": "user", "content": "Explain RAG in two sentences."}],
    stream=True,
)
for chunk in stream:
    print(chunk.choices[0].delta.content or "", end="", flush=True)

### Plain HTTP (curl, n8n, Postman)
Replace the URL with the one printed in step 4:

```bash
curl https://YOUR-TUNNEL.trycloudflare.com/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"model":"Qwen/Qwen2.5-3B-Instruct","messages":[{"role":"user","content":"Hi"}]}'
```

## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| Startup log shows `SIGINT` / server dies while loading | Server was stopped or started twice | Start once; don't interrupt the cell |
| `AssertionError: Error in memory profiling ... free memory increased` | An older vLLM process was still holding GPU memory and released it mid-startup | Kill everything (below), wait for `nvidia-smi` to show ~0 MiB, start once |
| `CUDA out of memory` | Not enough GPU memory | Lower `--max-model-len` to 2048 and `--gpu-memory-utilization` to 0.80 |
| Cloudflare 502 / 1033 | Tunnel is up but vLLM isn't ready | Check `!tail -n 30 vllm.log` for `Application startup complete` |
| `bfloat16 → float16` warnings | Expected: Qwen weights are bf16, T4 needs fp16 | Ignore |

Clean restart:
```python
!pkill -9 -f vllm
!pkill -9 -f EngineCore
!sleep 10
!nvidia-smi --query-gpu=memory.used --format=csv
```

## Limitations
- **Not 24/7:** free Colab sessions end after a few hours or when idle.
- **The URL changes** every time the tunnel restarts (a free Cloudflare account allows a named, stable tunnel).
- **No authentication** by default: anyone with the URL can use the GPU. Add `--api-key` before sharing.
- **Model license:** check the Qwen2.5-3B-Instruct model card on Hugging Face for terms before commercial use.